# LongFlow P0 — Stage 1: contrast-pair generation + activation capture (v2)

Runtime: **L4 GPU**. Spec: `docs/experiments/p0-steering.md` Stage 1; capture code: `src/steering/contrast_pairs.py`; scripts/lead-ins: `configs/p0_contrast.json`.

v2: the recorder is now position-aware — the 2026-07-06 calibration proved VibeVoice's CFG negative pass fires only on speech-frame steps (116 calls = 1 prefill + 60 positive + 55 negative for 61 tokens), so the positive stream is recovered from `cache_position` chains instead of a fixed calls-per-step constant.

In [ ]:
!nvidia-smi -L
%cd /content
!git clone https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

LongFlow is a private repo — paste a GitHub personal access token (fine-grained, read-only on Josh-E-S/LongFlow) into the hidden prompt. If `/content/LongFlow` already exists from earlier in the session, this pulls the latest instead.

In [ ]:
import os
from getpass import getpass
token = getpass("GitHub token: ")
if os.path.isdir("/content/LongFlow/.git"):
    !cd /content/LongFlow && git pull -q https://{token}@github.com/Josh-E-S/LongFlow.git main
else:
    !git clone -q https://{token}@github.com/Josh-E-S/LongFlow.git /content/LongFlow
del token
import sys
sys.path.insert(0, "/content/LongFlow")
import importlib
import src.steering.contrast_pairs as cp
importlib.reload(cp)
import json
CFG = json.load(open("/content/LongFlow/configs/p0_contrast.json"))
print(len(CFG["scripts"]), "scripts;", list(CFG["axes"]))

In [ ]:
import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor

MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

# Stage-0 carry-over #1: record the exact weights revision
import os
from huggingface_hub import snapshot_download
print("weights revision:", os.path.basename(os.path.realpath(snapshot_download(MODEL_ID))))

# Stage-0 carry-over #2: the actual solver step count
for attr in ("ddpm_inference_steps", "num_inference_steps", "inference_steps"):
    for obj in (model, model.config, getattr(model, "generation_config", None)):
        if obj is not None and hasattr(obj, attr):
            print(f"{type(obj).__name__}.{attr} =", getattr(obj, attr))
!grep -rn "inference_steps" /content/VibeVoice/demo/inference_from_file.py | head -5

VOICE = "/content/VibeVoice/demo/voices/en-Alice_woman.wav"
LAYERS = model.model.language_model.layers
tok = processor.tokenizer
FRAME_ID = tok.convert_tokens_to_ids("<|vision_pad|>")
START_ID = tok.convert_tokens_to_ids("<|vision_start|>")
END_ID = tok.convert_tokens_to_ids("<|vision_end|>")
print(len(LAYERS), "layers; special ids:", FRAME_ID, START_ID, END_ID)

## Calibration check (now automatic — this cell only VERIFIES)

Expect: `states` has exactly `n_gen - 1` columns (every generated token except token 0, whose state lives in the prefill), token indices `1..n_gen-1`, and ≥2 speech_end markers (per-turn boundaries).

In [ ]:
tiny_script = "Speaker 1: One short calibration sentence.\nSpeaker 1: And a second one to test turn boundaries.\n"
inputs = processor(text=[tiny_script], voice_samples=[[VOICE]], return_tensors="pt", padding=True)
inputs = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

rec = cp.LayerActivationRecorder(LAYERS)
with rec, torch.inference_mode():
    out = model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)

seq = out.sequences[0] if hasattr(out, "sequences") else out[0]
prompt_len = inputs["input_ids"].shape[1]
gen_ids = seq[prompt_len:].tolist()
states, tok_idx = rec.step_states(prompt_len)
print(f"hook calls {rec.num_calls} | generated {len(gen_ids)} | positive states {states.shape}")
assert states.shape[1] == len(gen_ids) - 1, "positive-stream recovery incomplete — STOP, report this"
ends = [i for i, t in enumerate(gen_ids) if t == END_ID]
mask = cp.target_frame_mask(gen_ids, tok_idx, frame_id=FRAME_ID, end_id=END_ID)
print(f"speech_end positions {ends} | target-region frames kept: {int(mask.sum())}")
TURN_MASKING = len(ends) >= 2
print("turn masking:", TURN_MASKING)

## Full capture loop — 80 generations

In [ ]:
import os
import time
import soundfile as sf
os.makedirs("/content/p0_audio", exist_ok=True)
K = 2
records, failures = [], []
t0 = time.time()

conditions = [(s, a, p) for s in CFG["scripts"] for a in CFG["axes"] for p in ("pos", "neg")]
for ci, (script_id, axis, pole) in enumerate(conditions):
    lead_in = CFG["axes"][axis][pole]
    text = f"Speaker 1: {lead_in}\nSpeaker 1: {CFG['scripts'][script_id]}\n"
    for k in range(K):
        try:
            inputs = processor(text=[text], voice_samples=[[VOICE]], return_tensors="pt", padding=True)
            inputs = {kk: (v.to("cuda") if hasattr(v, "to") else v) for kk, v in inputs.items()}
            rec = cp.LayerActivationRecorder(LAYERS)
            with rec, torch.inference_mode():
                out = model.generate(**inputs, tokenizer=processor.tokenizer, cfg_scale=1.3)
            seq = out.sequences[0] if hasattr(out, "sequences") else out[0]
            prompt_len = inputs["input_ids"].shape[1]
            gen_ids = seq[prompt_len:].tolist()
            states, tok_idx = rec.step_states(prompt_len)
            mask = cp.target_frame_mask(
                gen_ids, tok_idx, frame_id=FRAME_ID, end_id=END_ID if TURN_MASKING else None
            )
            if not TURN_MASKING:
                mask = cp.drop_leading_true(mask, 0.35)
            records.append(cp.pool_record(
                states, mask, script_id=script_id, axis=axis, pole=pole,
                sample_idx=k, num_calls_total=rec.num_calls,
            ))
            if hasattr(out, "speech_outputs") and out.speech_outputs and k == 0:
                wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
                sf.write(f"/content/p0_audio/{script_id}_{axis}_{pole}.wav", wav, 24000)
        except Exception as e:
            failures.append((script_id, axis, pole, k, repr(e)))
            print("FAIL:", script_id, axis, pole, k, repr(e)[:200])
    print(f"[{ci+1}/{len(conditions)}] {script_id}/{axis}/{pole}  {len(records)} records  {(time.time()-t0)/60:.1f} min")

cp.save_records(records, "/content/vectors.pt")
print(f"saved {len(records)} records, {len(failures)} failures")

## Honesty check (spec Stage 1.4) — listen to pole pairs

Same script, opposite poles. If you cannot HEAR a difference in affect, the extraction has no signal to find — note it and consider the Expresso fallback (`docs/resources.md` §4).

In [ ]:
from IPython.display import Audio, display
for sid in list(CFG["scripts"])[:3]:
    for axis in CFG["axes"]:
        for pole in ("pos", "neg"):
            print(sid, axis, pole)
            display(Audio(f"/content/p0_audio/{sid}_{axis}_{pole}.wav"))

## Save out (Colab disk is wiped!)

Download `vectors.pt` (→ `experiments/p0_steering/`, gitignored) and a few honesty-check wavs (→ `experiments/p0_steering/audio/`). Record in NOTES.md: weights revision, solver step count, honesty-check listening verdict, failure count.

In [ ]:
from google.colab import files
files.download("/content/vectors.pt")